# AegisOnco Phase 3 — Reproducible METABRIC Survival Benchmark

This notebook implements a **research-only** survival benchmark on the attached METABRIC table. It compares centralized, local-only, FedAvg, and FedProx training with deterministic patient-level splits inside each real METABRIC cohort. All reported final metrics use held-out test patients; validation patients are used only for model selection.

**Important scope:** METABRIC contains no paired radiology images. The `imaging` input below consists only of histologic, morphometric, receptor, and mutation-count variables and is explicitly a **proxy imaging-adjacent feature block**, not scan-derived imaging. This notebook does not estimate treatment effects, does not provide clinical recommendations, and is not validated for clinical deployment.


In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import random
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from IPython.display import display

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACT_VERSION = "v1.0.0"
ARTIFACT_ROOT = Path("/kaggle/working/aegisonco_artifacts") / ARTIFACT_VERSION

GENE_PANEL = [
    "tp53", "egfr", "kras", "erbb2", "pik3ca", "brca1", "brca2", "myc",
    "pten", "atm", "cdh1", "gata3", "akt1", "mtor", "braf", "notch1",
]
CLINICAL_FEATURES = [
    "age_at_diagnosis", "tumor_size", "lymph_nodes_examined_positive",
    "nottingham_prognostic_index", "tumor_stage",
]
PROXY_FEATURES = [
    "neoplasm_histologic_grade", "cellularity", "mutation_count",
    "er_status", "her2_status", "pr_status",
]
CELLULARITY_MAP = {"low": 1.0, "moderate": 2.0, "high": 3.0}
STATUS_MAP = {"negative": 0.0, "positive": 1.0}


def locate_metabric_csv() -> Path:
    preferred = Path(
        "/kaggle/input/breast-cancer-gene-expression-profiles-metabric/"
        "METABRIC_RNA_Mutation.csv"
    )
    if preferred.exists():
        return preferred
    candidates = sorted(Path("/kaggle/input").glob("**/*METABRIC*.csv"))
    if not candidates:
        raise FileNotFoundError(
            "Attach the METABRIC Kaggle dataset; no *METABRIC*.csv was found under /kaggle/input."
        )
    return candidates[0]


def resolve_column(df: pd.DataFrame, candidates: Sequence[str]) -> str:
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    raise KeyError(f"None of the required columns exist: {list(candidates)}")


def derive_event_indicator(df: pd.DataFrame) -> Tuple[pd.Series, Dict[str, object]]:
    """Return 1=death observed, auditing binary orientation against death status when available."""
    status_candidates = ["overall_survival_status", "os_status"]
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for candidate in status_candidates:
        if candidate in lookup:
            text = df[lookup[candidate]].astype("string").str.strip().str.lower()
            death = text.str.contains("deceased|dead|died|^1:", regex=True, na=False)
            alive = text.str.contains("living|alive|^0:", regex=True, na=False)
            unknown = ~(death | alive)
            if unknown.any():
                examples = text[unknown].value_counts(dropna=False).head(10).to_dict()
                raise ValueError(f"Unrecognized or missing survival-status categories: {examples}")
            event = death.astype(np.int64)
            return event, {"source": lookup[candidate], "orientation": "parsed and validated death-status text"}

    outcome_col = resolve_column(df, ["overall_survival"])
    numeric = pd.to_numeric(df[outcome_col], errors="coerce")
    values = set(numeric.dropna().unique().tolist())
    if not values.issubset({0, 1, 0.0, 1.0}):
        raise ValueError(f"{outcome_col} must be binary; observed values include {sorted(values)[:10]}")

    orientation = "1=death (METABRIC/cBioPortal convention)"
    event = numeric.astype("Int64")
    if "death_from_cancer" in lookup:
        death_text = df[lookup["death_from_cancer"]].astype(str).str.lower()
        known = death_text.str.contains("living|died|dead|deceased", regex=True)
        death = death_text.str.contains("died|dead|deceased", regex=True)
        valid = known & event.notna()
        if valid.any():
            agree_as_is = (event[valid].astype(int) == death[valid].astype(int)).mean()
            agree_flipped = ((1 - event[valid].astype(int)) == death[valid].astype(int)).mean()
            if agree_flipped > agree_as_is:
                event = 1 - event
                orientation = "0=death, selected by agreement with death_from_cancer"
            else:
                orientation = "1=death, validated against death_from_cancer"
            if max(agree_as_is, agree_flipped) < 0.75:
                raise ValueError("Could not reliably orient overall_survival against death_from_cancer.")
    if event.isna().any():
        raise ValueError("Missing overall-survival event indicators remain after filtering.")
    return event.astype(np.int64), {"source": outcome_col, "orientation": orientation}


DATA_PATH = locate_metabric_csv()
DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
raw_df.columns = [str(c).strip().lower() for c in raw_df.columns]
PATIENT_ID_COL = resolve_column(raw_df, ["patient_id", "sample_id"])
TIME_COL = resolve_column(raw_df, ["overall_survival_months"])
COHORT_COL = resolve_column(raw_df, ["cohort"])

required = set(CLINICAL_FEATURES + PROXY_FEATURES + GENE_PANEL + [PATIENT_ID_COL, TIME_COL, COHORT_COL])
missing = sorted(required.difference(raw_df.columns))
if missing:
    raise KeyError(f"METABRIC file is missing required columns: {missing}")

raw_df["event"] , EVENT_AUDIT = derive_event_indicator(raw_df)
raw_df[TIME_COL] = pd.to_numeric(raw_df[TIME_COL], errors="coerce")
analysis_df = raw_df.loc[
    raw_df[PATIENT_ID_COL].notna() & raw_df[TIME_COL].notna() & (raw_df[TIME_COL] > 0)
].copy()
if analysis_df[PATIENT_ID_COL].duplicated().any():
    duplicates = int(analysis_df[PATIENT_ID_COL].duplicated().sum())
    raise ValueError(f"Patient-level splitting requires unique IDs; found {duplicates} duplicate rows.")


def deterministic_stratified_split(
    frame: pd.DataFrame, seed: int, train_fraction: float = 0.70, val_fraction: float = 0.15
) -> Dict[str, pd.DataFrame]:
    """Deterministically split unique patients while approximately preserving event balance."""
    allocations = {"train": [], "validation": [], "test": []}
    for event_value in (0, 1):
        idx = frame.index[frame["event"] == event_value].to_numpy(copy=True)
        rng = np.random.default_rng(seed + 1009 * event_value)
        rng.shuffle(idx)
        n = len(idx)
        n_train = max(1, int(math.floor(train_fraction * n)))
        n_val = max(1, int(math.floor(val_fraction * n)))
        if n_train + n_val >= n:
            n_train, n_val = max(1, n - 2), 1
        allocations["train"].extend(idx[:n_train].tolist())
        allocations["validation"].extend(idx[n_train:n_train + n_val].tolist())
        allocations["test"].extend(idx[n_train + n_val:].tolist())

    result = {}
    for offset, (split_name, indices) in enumerate(allocations.items()):
        indices = np.asarray(indices)
        np.random.default_rng(seed + 37 * offset).shuffle(indices)
        result[split_name] = frame.loc[indices].copy().reset_index(drop=True)

    id_sets = {name: set(part[PATIENT_ID_COL].astype(str)) for name, part in result.items()}
    assert not (id_sets["train"] & id_sets["validation"])
    assert not (id_sets["train"] & id_sets["test"])
    assert not (id_sets["validation"] & id_sets["test"])
    assert sum(len(part) for part in result.values()) == len(frame)
    return result


cohort_values = sorted(pd.to_numeric(analysis_df[COHORT_COL], errors="raise").unique().tolist())
site_frames: Dict[str, Dict[str, pd.DataFrame]] = {}
for cohort in cohort_values:
    cohort_frame = analysis_df.loc[pd.to_numeric(analysis_df[COHORT_COL]) == cohort].copy()
    if len(cohort_frame) < 20:
        raise ValueError(f"Cohort {cohort} is too small for a three-way patient split ({len(cohort_frame)} rows).")
    site_name = f"METABRIC_Cohort_{int(cohort) if float(cohort).is_integer() else cohort}"
    site_seed = SEED + int(float(cohort) * 101)
    site_frames[site_name] = deterministic_stratified_split(cohort_frame, site_seed)

split_summary = {
    site: {
        split: {
            "patients": int(len(frame)),
            "events": int(frame["event"].sum()),
            "event_rate": float(frame["event"].mean()),
        }
        for split, frame in splits.items()
    }
    for site, splits in site_frames.items()
}

print(f"Loaded {len(analysis_df):,} unique METABRIC patients from {DATA_PATH}")
print(f"Event audit: {EVENT_AUDIT}")
print(f"Device: {DEVICE}; cohort partitions: {list(site_frames)}")
display(pd.DataFrame({(site, split): values for site, x in split_summary.items() for split, values in x.items()}).T)


In [ ]:
# Preprocessing is fitted once on the union of training partitions only, then frozen.
def raw_feature_blocks(frame: pd.DataFrame) -> Dict[str, np.ndarray]:
    clinical = frame[CLINICAL_FEATURES].apply(pd.to_numeric, errors="coerce").to_numpy(np.float64)
    genomics = frame[GENE_PANEL].apply(pd.to_numeric, errors="coerce").to_numpy(np.float64)
    proxy = np.column_stack([
        pd.to_numeric(frame["neoplasm_histologic_grade"], errors="coerce"),
        frame["cellularity"].astype(str).str.strip().str.lower().map(CELLULARITY_MAP),
        pd.to_numeric(frame["mutation_count"], errors="coerce"),
        frame["er_status"].astype(str).str.strip().str.lower().map(STATUS_MAP),
        frame["her2_status"].astype(str).str.strip().str.lower().map(STATUS_MAP),
        frame["pr_status"].astype(str).str.strip().str.lower().map(STATUS_MAP),
    ]).astype(np.float64)
    stage_present = frame["tumor_stage"].notna().to_numpy(np.float64)
    return {"clinical": clinical, "genomics": genomics, "imaging_proxy": proxy, "stage_present": stage_present}


def fit_block_statistics(matrix: np.ndarray) -> Dict[str, List[float]]:
    median = np.nanmedian(matrix, axis=0)
    if np.isnan(median).any():
        raise ValueError("At least one feature is entirely missing in pooled training data.")
    filled = np.where(np.isnan(matrix), median, matrix)
    mean = filled.mean(axis=0)
    scale = filled.std(axis=0)
    scale = np.where(scale < 1e-8, 1.0, scale)
    return {"median": median.tolist(), "mean": mean.tolist(), "scale": scale.tolist()}


pooled_train_frame = pd.concat([splits["train"] for splits in site_frames.values()], ignore_index=True)
pooled_train_raw = raw_feature_blocks(pooled_train_frame)
PREPROCESSING = {
    block: fit_block_statistics(pooled_train_raw[block])
    for block in ("clinical", "genomics", "imaging_proxy")
}


def apply_block(matrix: np.ndarray, stats: Mapping[str, Sequence[float]]) -> np.ndarray:
    median = np.asarray(stats["median"], dtype=np.float64)
    mean = np.asarray(stats["mean"], dtype=np.float64)
    scale = np.asarray(stats["scale"], dtype=np.float64)
    return ((np.where(np.isnan(matrix), median, matrix) - mean) / scale).astype(np.float32)


def transform_frame(frame: pd.DataFrame) -> Dict[str, object]:
    blocks = raw_feature_blocks(frame)
    return {
        "clinical": torch.from_numpy(apply_block(blocks["clinical"], PREPROCESSING["clinical"])),
        "clinical_mask": torch.from_numpy(blocks["stage_present"].astype(np.float32)),
        "genomics": torch.from_numpy(apply_block(blocks["genomics"], PREPROCESSING["genomics"])),
        "imaging": torch.from_numpy(apply_block(blocks["imaging_proxy"], PREPROCESSING["imaging_proxy"])),
        "time": torch.tensor(frame[TIME_COL].to_numpy(np.float32)),
        "event": torch.tensor(frame["event"].to_numpy(np.float32)),
        "patient_id": frame[PATIENT_ID_COL].astype(str).tolist(),
    }


tensor_splits = {
    site: {split: transform_frame(frame) for split, frame in splits.items()}
    for site, splits in site_frames.items()
}


def concatenate_data(parts: Iterable[Mapping[str, object]]) -> Dict[str, object]:
    parts = list(parts)
    tensor_keys = ["clinical", "clinical_mask", "genomics", "imaging", "time", "event"]
    result = {key: torch.cat([part[key] for part in parts], dim=0) for key in tensor_keys}
    result["patient_id"] = [patient for part in parts for patient in part["patient_id"]]
    return result


pooled = {
    split: concatenate_data([splits[split] for splits in tensor_splits.values()])
    for split in ("train", "validation", "test")
}

preprocessing_schema = {
    "artifact_version": ARTIFACT_VERSION,
    "fit_scope": "pooled training partitions only; frozen for validation and test",
    "random_seed": SEED,
    "dataset_sha256": DATA_SHA256,
    "environment": {
        "python": os.sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pytorch": torch.__version__,
        "device": str(DEVICE),
    },
    "patient_id_column": PATIENT_ID_COL,
    "time_column": TIME_COL,
    "event_definition": EVENT_AUDIT,
    "site_definition": f"simulated federated partition for each distinct value of {COHORT_COL}; cohort labels are not hospitals",
    "features": {
        "clinical": CLINICAL_FEATURES,
        "clinical_mask": ["tumor_stage_present"],
        "genomics": GENE_PANEL,
        "imaging_proxy": PROXY_FEATURES,
    },
    "categorical_maps": {"cellularity": CELLULARITY_MAP, "receptor_status": STATUS_MAP},
    "statistics": PREPROCESSING,
    "missing_value_policy": "training median imputation followed by training mean/SD standardization",
    "proxy_imaging_warning": "Histology/receptor/mutation variables only; no pixels, scans, or radiology features.",
    "split_summary": split_summary,
}

# Leakage guard: preprocessing row count must equal only the pooled training count.
assert len(pooled_train_frame) == len(pooled["train"]["time"])
assert set(pooled["train"]["patient_id"]).isdisjoint(pooled["validation"]["patient_id"])
assert set(pooled["train"]["patient_id"]).isdisjoint(pooled["test"]["patient_id"])
print(f"Fitted preprocessing on {len(pooled_train_frame):,} training patients only.")


In [ ]:
class METABRICSurvivalDataset(Dataset):
    def __init__(self, data: Mapping[str, object]):
        self.data = data

    def __len__(self) -> int:
        return len(self.data["time"])

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        return {key: self.data[key][index] for key in (
            "clinical", "clinical_mask", "genomics", "imaging", "time", "event"
        )}


class SurvivalFusionTransformer(nn.Module):
    """Compact multimodal risk model sized for Kaggle CPU/GPU limits."""
    def __init__(self, embed_dim: int = 32, num_heads: int = 4, num_layers: int = 1):
        super().__init__()
        # The stage-observed mask is retained as a feature; missing stage does not hide other clinical values.
        self.clinical_proj = nn.Sequential(nn.Linear(6, embed_dim), nn.LayerNorm(embed_dim), nn.GELU())
        self.genomic_proj = nn.Sequential(nn.Linear(16, embed_dim), nn.LayerNorm(embed_dim), nn.GELU())
        self.proxy_proj = nn.Sequential(nn.Linear(6, embed_dim), nn.LayerNorm(embed_dim), nn.GELU())
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 2,
            dropout=0.10, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers, enable_nested_tensor=False)
        self.modality_embedding = nn.Parameter(torch.zeros(1, 3, embed_dim))
        nn.init.normal_(self.modality_embedding, std=0.02)
        self.risk_head = nn.Sequential(
            nn.Linear(embed_dim * 3, 32), nn.GELU(), nn.Dropout(0.10), nn.Linear(32, 1)
        )

    def forward(
        self, clinical: torch.Tensor, clinical_mask: torch.Tensor,
        genomics: torch.Tensor, imaging: torch.Tensor,
    ) -> torch.Tensor:
        mask_feature = clinical_mask.reshape(-1, 1).to(clinical.dtype)
        clinical_with_mask = torch.cat([clinical, mask_feature], dim=1)
        tokens = torch.stack([
            self.clinical_proj(clinical_with_mask),
            self.genomic_proj(genomics),
            self.proxy_proj(imaging),
        ], dim=1)
        encoded = self.encoder(tokens + self.modality_embedding)
        return self.risk_head(encoded.flatten(start_dim=1)).squeeze(-1)


class CoxPartialLikelihoodLoss(nn.Module):
    """Negative Cox partial log likelihood with exact risk sets and Breslow ties."""
    def forward(self, log_risk: torch.Tensor, times: torch.Tensor, events: torch.Tensor) -> torch.Tensor:
        log_risk = log_risk.reshape(-1)
        times = times.reshape(-1)
        events = events.reshape(-1).to(log_risk.dtype)
        if not (len(log_risk) == len(times) == len(events)):
            raise ValueError("log_risk, times, and events must have equal lengths")
        if events.sum() <= 0:
            return log_risk.sum() * 0.0

        # Descending times put every subject with T_j >= t in a prefix risk set.
        order = torch.argsort(times, descending=True, stable=True)
        sorted_time = times[order]
        sorted_risk = log_risk[order]
        sorted_event = events[order]
        log_prefix_risk = torch.logcumsumexp(sorted_risk, dim=0)

        # For a tied time, Breslow uses the risk set after all members of that tie enter.
        _, counts = torch.unique_consecutive(sorted_time, return_counts=True)
        group_ends = torch.cumsum(counts, dim=0) - 1
        group_ids = torch.repeat_interleave(
            torch.arange(len(counts), device=log_risk.device), counts
        )
        deaths = torch.zeros(len(counts), dtype=log_risk.dtype, device=log_risk.device)
        event_risk_sum = torch.zeros_like(deaths)
        deaths.scatter_add_(0, group_ids, sorted_event)
        event_risk_sum.scatter_add_(0, group_ids, sorted_risk * sorted_event)
        has_event = deaths > 0
        partial_log_likelihood = (
            event_risk_sum[has_event] - deaths[has_event] * log_prefix_risk[group_ends][has_event]
        ).sum()
        return -partial_log_likelihood / events.sum()


# Risk-set regression tests: these fail under the old reversed-cumsum construction.
criterion = CoxPartialLikelihoodLoss()
zero_risk = torch.zeros(3, requires_grad=True)
untied = criterion(zero_risk, torch.tensor([3.0, 2.0, 1.0]), torch.ones(3))
expected_untied = (math.log(1.0) + math.log(2.0) + math.log(3.0)) / 3.0
assert torch.allclose(untied, torch.tensor(expected_untied), atol=1e-6)
tied = criterion(zero_risk, torch.tensor([2.0, 2.0, 1.0]), torch.ones(3))
expected_tied = (2.0 * math.log(2.0) + math.log(3.0)) / 3.0
assert torch.allclose(tied, torch.tensor(expected_tied), atol=1e-6)
print("Cox risk-set and Breslow-tie checks passed.")


In [ ]:
def to_device(data: Mapping[str, object], device: torch.device = DEVICE) -> Dict[str, torch.Tensor]:
    return {key: data[key].to(device) for key in (
        "clinical", "clinical_mask", "genomics", "imaging", "time", "event"
    )}


def predict_scores(model: nn.Module, data: Mapping[str, object]) -> np.ndarray:
    model.eval()
    x = to_device(data)
    with torch.no_grad():
        scores = model(x["clinical"], x["clinical_mask"], x["genomics"], x["imaging"])
    return scores.detach().cpu().numpy().astype(np.float64)


def concordance_index(scores: np.ndarray, times: np.ndarray, events: np.ndarray) -> float:
    """Harrell C-index; higher score means higher risk and earlier observed death."""
    scores, times, events = map(np.asarray, (scores, times, events))
    concordant = 0.0
    comparable = 0
    for i in range(len(times)):
        eligible = (times > times[i]) & (events[i] == 1)
        eligible[i] = False
        n = int(eligible.sum())
        if n:
            comparable += n
            concordant += float((scores[i] > scores[eligible]).sum())
            concordant += 0.5 * float((scores[i] == scores[eligible]).sum())
    return float(concordant / comparable) if comparable else float("nan")


def breslow_baseline_hazard(
    scores: np.ndarray, times: np.ndarray, events: np.ndarray
) -> Dict[str, List[float]]:
    """Estimate H0(t) from training patients only using Breslow event-time increments."""
    scores, times, events = map(np.asarray, (scores, times, events))
    event_times = np.sort(np.unique(times[events == 1]))
    increments = []
    for event_time in event_times:
        deaths = int(((times == event_time) & (events == 1)).sum())
        denominator = np.exp(np.clip(scores[times >= event_time], -30.0, 30.0)).sum()
        increments.append(float(deaths / max(denominator, 1e-12)))
    cumulative = np.cumsum(increments)
    return {
        "time_months": event_times.astype(float).tolist(),
        "hazard_increment": np.asarray(increments, dtype=float).tolist(),
        "cumulative_hazard": cumulative.astype(float).tolist(),
        "estimator": "Breslow; fitted on training patients only",
    }


def cumulative_hazard_at(baseline: Mapping[str, Sequence[float]], horizons: Sequence[float]) -> np.ndarray:
    event_times = np.asarray(baseline["time_months"], dtype=float)
    cumulative = np.asarray(baseline["cumulative_hazard"], dtype=float)
    indices = np.searchsorted(event_times, np.asarray(horizons, dtype=float), side="right") - 1
    return np.where(indices >= 0, cumulative[np.maximum(indices, 0)], 0.0)


def survival_probabilities(
    scores: np.ndarray, baseline: Mapping[str, Sequence[float]], horizons: Sequence[float]
) -> np.ndarray:
    h0 = cumulative_hazard_at(baseline, horizons)
    return np.exp(-np.exp(np.clip(scores, -30.0, 30.0))[:, None] * h0[None, :])


def censoring_km(reference_times: np.ndarray, reference_events: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    times = np.asarray(reference_times, dtype=float)
    censored = 1 - np.asarray(reference_events, dtype=int)
    unique_times = np.sort(np.unique(times))
    survival, values = 1.0, []
    for current in unique_times:
        at_risk = int((times >= current).sum())
        censorings = int(((times == current) & (censored == 1)).sum())
        if at_risk:
            survival *= 1.0 - censorings / at_risk
        values.append(survival)
    return unique_times, np.asarray(values, dtype=float)


def step_survival_at(km_times: np.ndarray, km_values: np.ndarray, query, left_limit: bool = False):
    side = "left" if left_limit else "right"
    indices = np.searchsorted(km_times, query, side=side) - 1
    result = np.ones(np.asarray(query).shape, dtype=float)
    valid = indices >= 0
    result[valid] = km_values[indices[valid]]
    return np.clip(result, 1e-6, 1.0)


def ipcw_brier_scores(
    predicted_survival: np.ndarray,
    times: np.ndarray,
    events: np.ndarray,
    horizons: Sequence[float],
    censor_reference_times: np.ndarray,
    censor_reference_events: np.ndarray,
) -> List[float]:
    """IPCW Brier scores using the training-set censoring distribution."""
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    km_t, km_g = censoring_km(censor_reference_times, censor_reference_events)
    output = []
    for column, horizon in enumerate(horizons):
        pred = predicted_survival[:, column]
        died = (times <= horizon) & (events == 1)
        survived = times > horizon
        weights = np.zeros(len(times), dtype=float)
        if died.any():
            weights[died] = 1.0 / step_survival_at(km_t, km_g, times[died], left_limit=True)
        if survived.any():
            weights[survived] = 1.0 / step_survival_at(
                km_t, km_g, np.full(int(survived.sum()), horizon), left_limit=False
            )
        target = survived.astype(float)
        output.append(float(np.mean(weights * (target - pred) ** 2)))
    return output


def km_event_survival_at(times: np.ndarray, events: np.ndarray, horizon: float) -> float:
    times, events = np.asarray(times, float), np.asarray(events, int)
    survival = 1.0
    for current in np.sort(np.unique(times[times <= horizon])):
        at_risk = int((times >= current).sum())
        deaths = int(((times == current) & (events == 1)).sum())
        if at_risk:
            survival *= 1.0 - deaths / at_risk
    return float(survival)


def calibration_table(
    predicted_survival: np.ndarray, times: np.ndarray, events: np.ndarray, horizons: Sequence[float]
) -> List[Dict[str, object]]:
    """Quartile calibration summaries suitable for reliability plotting downstream."""
    rows = []
    for column, horizon in enumerate(horizons):
        risk = 1.0 - predicted_survival[:, column]
        labels = pd.qcut(pd.Series(risk), q=4, labels=False, duplicates="drop").to_numpy()
        for group in sorted(pd.Series(labels).dropna().unique().astype(int).tolist()):
            use = labels == group
            rows.append({
                "horizon_months": float(horizon),
                "risk_group": int(group + 1),
                "patients": int(use.sum()),
                "mean_predicted_survival": float(predicted_survival[use, column].mean()),
                "kaplan_meier_survival": km_event_survival_at(times[use], events[use], float(horizon)),
            })
    return rows


def evaluate_predictions(
    scores: np.ndarray,
    predicted_survival: np.ndarray,
    data: Mapping[str, object],
    horizons: Sequence[float],
    censor_reference: Mapping[str, object],
) -> Dict[str, object]:
    times = data["time"].cpu().numpy().astype(float)
    events = data["event"].cpu().numpy().astype(int)
    ref_times = censor_reference["time"].cpu().numpy().astype(float)
    ref_events = censor_reference["event"].cpu().numpy().astype(int)
    brier = ipcw_brier_scores(predicted_survival, times, events, horizons, ref_times, ref_events)
    return {
        "patients": int(len(times)),
        "events": int(events.sum()),
        "c_index": concordance_index(scores, times, events),
        "brier_by_horizon": {str(float(h)): float(value) for h, value in zip(horizons, brier)},
        "mean_brier": float(np.mean(brier)),
        "calibration": calibration_table(predicted_survival, times, events, horizons),
    }


train_event_times = pooled["train"]["time"][pooled["train"]["event"] == 1].numpy()
HORIZONS = np.quantile(train_event_times, [0.25, 0.50, 0.75]).round(3).tolist()
print(f"Evaluation horizons selected from pooled training event times only: {HORIZONS} months")


In [ ]:
# Runtime-conscious benchmark configuration. Full-batch optimization preserves complete site risk sets.
CONFIG = {
    "communication_rounds": 8,
    "local_epochs_per_round": 1,
    "standalone_epochs": 8,
    "learning_rate": 2e-3,
    "weight_decay": 1e-4,
    "fedprox_mu": 0.01,
    "gradient_clip": 2.0,
}


def fresh_model() -> SurvivalFusionTransformer:
    model = SurvivalFusionTransformer().to(DEVICE)
    model.load_state_dict(copy.deepcopy(INITIAL_STATE))
    return model


def cpu_state(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def train_full_risk_epochs(
    model: nn.Module,
    data: Mapping[str, object],
    epochs: int,
    proximal_anchor: Mapping[str, torch.Tensor] | None = None,
    mu: float = 0.0,
    optimizer: optim.Optimizer | None = None,
) -> List[float]:
    """Train on one complete partition so each Cox denominator sees its full risk set."""
    x = to_device(data)
    if optimizer is None:
        optimizer = optim.AdamW(
            model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"]
        )
    anchor = None
    if proximal_anchor is not None:
        anchor = {name: value.to(DEVICE) for name, value in proximal_anchor.items()}
    losses = []
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        score = model(x["clinical"], x["clinical_mask"], x["genomics"], x["imaging"])
        loss = criterion(score, x["time"], x["event"])
        if anchor is not None and mu > 0:
            penalty = sum(
                (parameter - anchor[name]).pow(2).sum()
                for name, parameter in model.named_parameters()
            )
            loss = loss + 0.5 * mu * penalty
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip"])
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return losses


def validation_c_indices(model: nn.Module) -> Dict[str, float]:
    metrics = {}
    for site, splits in tensor_splits.items():
        data = splits["validation"]
        metrics[site] = concordance_index(
            predict_scores(model, data), data["time"].numpy(), data["event"].numpy()
        )
    metrics["global_pooled"] = concordance_index(
        predict_scores(model, pooled["validation"]),
        pooled["validation"]["time"].numpy(),
        pooled["validation"]["event"].numpy(),
    )
    return metrics


def weighted_average_states(
    states_and_sizes: Sequence[Tuple[Mapping[str, torch.Tensor], int]]
) -> Dict[str, torch.Tensor]:
    total = float(sum(size for _, size in states_and_sizes))
    output = {}
    for name in states_and_sizes[0][0]:
        first = states_and_sizes[0][0][name]
        if first.dtype.is_floating_point:
            output[name] = sum(state[name] * (size / total) for state, size in states_and_sizes)
        else:
            output[name] = first.clone()
    return output


# Identical deterministic initialization makes method comparisons less initialization-dependent.
torch.manual_seed(SEED)
INITIAL_STATE = cpu_state(SurvivalFusionTransformer())
training_history: Dict[str, object] = {}

# Centralized baseline: pooled training patients; selection uses pooled validation only.
centralized_model = fresh_model()
centralized_optimizer = optim.AdamW(
    centralized_model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"]
)
centralized_best_state, centralized_best_c = cpu_state(centralized_model), -np.inf
centralized_history = []
for epoch in range(1, CONFIG["standalone_epochs"] + 1):
    loss = train_full_risk_epochs(
        centralized_model, pooled["train"], epochs=1, optimizer=centralized_optimizer
    )[0]
    val_metrics = validation_c_indices(centralized_model)
    centralized_history.append({"epoch": epoch, "train_loss": loss, "validation_c_index": val_metrics})
    if val_metrics["global_pooled"] > centralized_best_c:
        centralized_best_c = val_metrics["global_pooled"]
        centralized_best_state = cpu_state(centralized_model)
centralized_model.load_state_dict(centralized_best_state)
training_history["centralized"] = centralized_history

# Local-only baselines: one independent model per cohort partition, selected on that partition's validation split.
local_models: Dict[str, nn.Module] = {}
local_history: Dict[str, List[Dict[str, object]]] = {}
for site, splits in tensor_splits.items():
    model = fresh_model()
    local_optimizer = optim.AdamW(
        model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"]
    )
    best_state, best_c = cpu_state(model), -np.inf
    history = []
    for epoch in range(1, CONFIG["standalone_epochs"] + 1):
        loss = train_full_risk_epochs(
            model, splits["train"], epochs=1, optimizer=local_optimizer
        )[0]
        scores = predict_scores(model, splits["validation"])
        val_c = concordance_index(scores, splits["validation"]["time"].numpy(), splits["validation"]["event"].numpy())
        history.append({"epoch": epoch, "train_loss": loss, "validation_c_index": val_c})
        if val_c > best_c:
            best_c, best_state = val_c, cpu_state(model)
    model.load_state_dict(best_state)
    local_models[site] = model
    local_history[site] = history
training_history["local_only"] = local_history


def run_federated(method: str, mu: float) -> Tuple[nn.Module, List[Dict[str, object]], float]:
    global_model = fresh_model()
    best_state, best_c = cpu_state(global_model), -np.inf
    rounds = []
    for round_number in range(1, CONFIG["communication_rounds"] + 1):
        anchor = cpu_state(global_model)
        updates = []
        client_losses = {}
        for site, splits in tensor_splits.items():
            local_model = copy.deepcopy(global_model).to(DEVICE)
            losses = train_full_risk_epochs(
                local_model, splits["train"], CONFIG["local_epochs_per_round"],
                proximal_anchor=anchor if mu > 0 else None, mu=mu,
            )
            client_losses[site] = losses
            updates.append((cpu_state(local_model), len(splits["train"]["time"])))
            del local_model
        global_model.load_state_dict(weighted_average_states(updates))
        val_metrics = validation_c_indices(global_model)
        rounds.append({
            "round": round_number,
            "client_train_losses": client_losses,
            "validation_c_index": val_metrics,
        })
        if val_metrics["global_pooled"] > best_c:
            best_c, best_state = val_metrics["global_pooled"], cpu_state(global_model)
        print(
            f"{method} round {round_number:02d}/{CONFIG['communication_rounds']}: "
            f"held-out validation global C-index={val_metrics['global_pooled']:.4f}"
        )
    global_model.load_state_dict(best_state)
    return global_model, rounds, float(best_c)


fedavg_model, fedavg_history, fedavg_best_c = run_federated("FedAvg", mu=0.0)
fedprox_model, fedprox_history, fedprox_best_c = run_federated("FedProx", mu=CONFIG["fedprox_mu"])
training_history["fedavg"] = fedavg_history
training_history["fedprox"] = fedprox_history

validation_selection = {
    "centralized": float(centralized_best_c),
    "fedavg": float(fedavg_best_c),
    "fedprox": float(fedprox_best_c),
    "local_only_by_site": {
        site: float(max(row["validation_c_index"] for row in rows))
        for site, rows in local_history.items()
    },
}
print("Training complete. Test partitions have not been used for optimization or model selection.")
print(json.dumps(validation_selection, indent=2))


In [ ]:
def evaluate_global_method(model: nn.Module) -> Tuple[Dict[str, object], Dict[str, object]]:
    train_scores = predict_scores(model, pooled["train"])
    baseline = breslow_baseline_hazard(
        train_scores, pooled["train"]["time"].numpy(), pooled["train"]["event"].numpy()
    )
    result = {"per_site": {}}
    site_scores, site_survival = [], []
    for site, splits in tensor_splits.items():
        scores = predict_scores(model, splits["test"])
        survival = survival_probabilities(scores, baseline, HORIZONS)
        result["per_site"][site] = evaluate_predictions(
            scores, survival, splits["test"], HORIZONS, splits["train"]
        )
        site_scores.append(scores)
        site_survival.append(survival)
    all_scores = np.concatenate(site_scores)
    all_survival = np.concatenate(site_survival, axis=0)
    result["global_pooled"] = evaluate_predictions(
        all_scores, all_survival, pooled["test"], HORIZONS, pooled["train"]
    )
    return result, baseline


def evaluate_local_only() -> Tuple[Dict[str, object], Dict[str, object]]:
    result = {"per_site": {}}
    baselines, site_scores, site_survival = {}, [], []
    for site, model in local_models.items():
        splits = tensor_splits[site]
        train_scores = predict_scores(model, splits["train"])
        baseline = breslow_baseline_hazard(
            train_scores, splits["train"]["time"].numpy(), splits["train"]["event"].numpy()
        )
        baselines[site] = baseline
        scores = predict_scores(model, splits["test"])
        survival = survival_probabilities(scores, baseline, HORIZONS)
        result["per_site"][site] = evaluate_predictions(
            scores, survival, splits["test"], HORIZONS, splits["train"]
        )
        site_scores.append(scores)
        site_survival.append(survival)
    # Pooled local-only C-index combines site-specific model scores; cross-site scale is not guaranteed.
    result["global_pooled"] = evaluate_predictions(
        np.concatenate(site_scores), np.concatenate(site_survival, axis=0),
        pooled["test"], HORIZONS, pooled["train"],
    )
    result["global_pooled"]["interpretation_note"] = (
        "Each test patient was scored by its own site's model. Cross-site score scales may differ."
    )
    return result, baselines


held_out_test: Dict[str, object] = {}
baselines: Dict[str, object] = {}
for method, model in {
    "centralized": centralized_model,
    "fedavg": fedavg_model,
    "fedprox": fedprox_model,
}.items():
    held_out_test[method], baselines[method] = evaluate_global_method(model)
held_out_test["local_only"], baselines["local_only"] = evaluate_local_only()

summary_rows = []
for method, metrics in held_out_test.items():
    summary_rows.append({
        "method": method,
        "site": "GLOBAL_POOLED",
        "test_c_index": metrics["global_pooled"]["c_index"],
        "test_mean_brier": metrics["global_pooled"]["mean_brier"],
    })
    for site, site_metrics in metrics["per_site"].items():
        summary_rows.append({
            "method": method,
            "site": site,
            "test_c_index": site_metrics["c_index"],
            "test_mean_brier": site_metrics["mean_brier"],
        })
summary_table = pd.DataFrame(summary_rows)
print("FINAL HELD-OUT TEST METRICS (never training loaders)")
display(summary_table.pivot(index="site", columns="method", values=["test_c_index", "test_mean_brier"]).round(4))
print("Calibration-ready quartile rows and horizon-specific IPCW Brier scores are retained in experiment_metrics.json.")


In [ ]:
def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def write_json(path: Path, payload: Mapping[str, object]) -> None:
    path.write_text(
        json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False),
        encoding="utf-8",
    )


def patient_split_fingerprint() -> str:
    entries = []
    for site, splits in site_frames.items():
        for split, frame in splits.items():
            entries.extend(f"{site}|{split}|{patient}" for patient in frame[PATIENT_ID_COL].astype(str))
    return hashlib.sha256("\n".join(sorted(entries)).encode("utf-8")).hexdigest()


# Select one deployable global model by validation C-index only; local-only has no single global state.
global_models = {
    "centralized": centralized_model,
    "fedavg": fedavg_model,
    "fedprox": fedprox_model,
}
selected_method = max(global_models, key=lambda name: validation_selection[name])
selected_model = copy.deepcopy(global_models[selected_method]).cpu().eval()
selected_baseline = baselines[selected_method]

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
torch.save(cpu_state(selected_model), ARTIFACT_ROOT / "model_state_dict.pt")

preprocessing_schema["model"] = {
    "class": "SurvivalFusionTransformer",
    "clinical_features": 5,
    "clinical_mask_features": 1,
    "genomic_features": 16,
    "imaging_proxy_features": 6,
    "embed_dim": 32,
    "num_heads": 4,
    "num_layers": 1,
    "output": "scalar log relative risk",
}
preprocessing_schema["patient_split_sha256"] = patient_split_fingerprint()
write_json(ARTIFACT_ROOT / "preprocessing_schema.json", preprocessing_schema)
write_json(
    ARTIFACT_ROOT / "baseline_hazard.json",
    {
        "artifact_version": ARTIFACT_VERSION,
        "selected_method": selected_method,
        "time_unit": "months",
        **selected_baseline,
    },
)


class ONNXSurvivalWrapper(nn.Module):
    """Stable four-input deployment signature returning [batch, 1] log-risk."""
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, clinical, clinical_mask, genomics, imaging):
        return self.model(clinical, clinical_mask, genomics, imaging).unsqueeze(1)


onnx_status = {"attempted": True, "succeeded": False, "path": None, "error": None}
try:
    wrapper = ONNXSurvivalWrapper(selected_model).eval()
    onnx_path = ARTIFACT_ROOT / "model.onnx"
    torch.onnx.export(
        wrapper,
        (
            torch.zeros(1, 5, dtype=torch.float32),
            torch.ones(1, dtype=torch.float32),
            torch.zeros(1, 16, dtype=torch.float32),
            torch.zeros(1, 6, dtype=torch.float32),
        ),
        onnx_path,
        input_names=["clinical", "clinical_mask", "genomics", "imaging_proxy"],
        output_names=["log_relative_risk"],
        dynamic_axes={
            "clinical": {0: "batch"}, "clinical_mask": {0: "batch"},
            "genomics": {0: "batch"}, "imaging_proxy": {0: "batch"},
            "log_relative_risk": {0: "batch"},
        },
        opset_version=17,
        do_constant_folding=True,
    )
    onnx_status.update({"succeeded": True, "path": onnx_path.name})
except Exception as exc:
    onnx_status["error"] = f"{type(exc).__name__}: {exc}"

experiment_metrics = {
    "artifact_version": ARTIFACT_VERSION,
    "seed": SEED,
    "device": str(DEVICE),
    "configuration": CONFIG,
    "evaluation_horizons_months": HORIZONS,
    "split_summary": split_summary,
    "validation_selection": validation_selection,
    "training_history": training_history,
    "held_out_test": held_out_test,
    "selected_global_method": selected_method,
    "onnx_export": onnx_status,
    "metric_notes": {
        "c_index": "Harrell C-index; higher model output means higher risk.",
        "brier": "IPCW Brier score using censoring KM estimated from the corresponding training partition.",
        "calibration": "Predicted survival versus Kaplan-Meier survival in test-risk quartiles.",
        "test_use": "Test partitions were evaluated once after validation-based selection.",
    },
}
write_json(ARTIFACT_ROOT / "experiment_metrics.json", experiment_metrics)

model_card = f"""# AegisOnco METABRIC Survival Model Card ({ARTIFACT_VERSION})

## Intended use
Research benchmarking of centralized and federated Cox risk models on METABRIC. **Not for diagnosis, treatment selection, prognosis delivery, or any clinical decision.** External and prospective validation has not been performed.

## Selected model
`{selected_method}` selected using pooled held-out validation C-index ({validation_selection[selected_method]:.4f}); test data were not used for selection. Output is a scalar Cox log-relative-risk. Absolute survival uses the training-only Breslow baseline in `baseline_hazard.json`.

## Inputs
Five standardized clinical values plus a tumor-stage-observed flag; 16 METABRIC mRNA z-scores; and six standardized histology/receptor/mutation variables. The latter are a **proxy imaging-adjacent block only**. There are no images, pixels, scans, or radiology-derived features.

## Evaluation
Per-site and pooled held-out C-index, horizon-specific IPCW Brier scores, and calibration quartile rows are in `experiment_metrics.json`. Cohort membership can encode batch, temporal, and case-mix effects. Local-only pooled scores require caution because independent site models need not share a common score scale.

## Limitations
Retrospective METABRIC data; modest site sizes; overall-survival endpoint; missing-data median imputation; proportional-hazards assumptions; no fairness or subgroup-safety validation; no competing-risk model; no causal treatment effects; and no clinical calibration outside this dataset. The model must remain research-only.

## Reproducibility
Seed `{SEED}`; deterministic patient split fingerprint `{preprocessing_schema['patient_split_sha256']}`. ONNX export success: `{onnx_status['succeeded']}`.
"""
(ARTIFACT_ROOT / "MODEL_CARD.md").write_text(model_card, encoding="utf-8")

data_card = f"""# METABRIC Data Card for AegisOnco Phase 3 ({ARTIFACT_VERSION})

## Source and unit of analysis
Attached Kaggle METABRIC RNA/mutation CSV; one row per unique patient after excluding missing/non-positive overall-survival time. Dataset `cohort` values define simulated federated partitions, not independent hospitals. Event audit: `{EVENT_AUDIT}`.

## Splitting and leakage controls
Each cohort is split independently and deterministically into approximately 70% train, 15% validation, and 15% test, stratified by observed death. Patient IDs are disjoint. Imputation medians, means, and scales are fitted only on pooled training partitions and then frozen. Validation selects checkpoints; test is used only for final reporting.

## Modalities
Clinical variables and a 16-gene expression panel are accompanied by histologic grade, cellularity, mutation count, and receptor status. These six fields are **not imaging data**; they are labeled proxy imaging-adjacent variables to preserve an interface shape without claiming scan evidence.

## Known limitations
METABRIC is retrospective and not representative of every geography, ancestry, care pathway, or contemporary treatment setting. Missingness mechanisms are not proven to be MNAR. Overall survival includes non-cancer mortality, and cohort labels are not independent randomized hospitals. Dataset licensing and original publication terms remain applicable.
"""
(ARTIFACT_ROOT / "DATA_CARD.md").write_text(data_card, encoding="utf-8")

manifest_files = [
    "model_state_dict.pt", "preprocessing_schema.json", "baseline_hazard.json",
    "experiment_metrics.json", "MODEL_CARD.md", "DATA_CARD.md",
]
if onnx_status["succeeded"]:
    manifest_files.append("model.onnx")
manifest = {
    "artifact_version": ARTIFACT_VERSION,
    "selected_method": selected_method,
    "files": {
        name: hashlib.sha256((ARTIFACT_ROOT / name).read_bytes()).hexdigest()
        for name in manifest_files
    },
    "onnx_export": onnx_status,
}
write_json(ARTIFACT_ROOT / "manifest.json", manifest)

# Strictly validate every JSON artifact immediately after writing it.
for json_path in ARTIFACT_ROOT.glob("*.json"):
    json.loads(json_path.read_text(encoding="utf-8"))

print(f"Versioned artifact bundle written to: {ARTIFACT_ROOT}")
print(f"Selected method: {selected_method}; ONNX export: {onnx_status}")
print("Artifacts:", sorted(path.name for path in ARTIFACT_ROOT.iterdir()))


## Interpretation and release constraints

- Final C-index and Brier/calibration outputs above are from held-out test patients, never training loaders.
- Federated round histories contain measured client losses and held-out validation C-indices, not fabricated example values.
- The baseline cumulative hazard is estimated from the selected model's pooled training predictions with the Breslow estimator; it is not a synthetic curve.
- The proxy `imaging` block is histology/receptor/mutation metadata only. It must not be described as radiology or scan-derived imaging.
- No treatment counterfactual, “digital twin,” confidence guarantee, clinical workflow, or production-readiness claim is supported. This benchmark is research-only.
